# 5.1 Markowitz 均值-方差优化

## 学习目标
- 理解现代投资组合理论（MPT）的核心思想
- 用 Python 计算有效前沿
- 找到最大夏普比率组合和最小方差组合

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from scipy.optimize import minimize

plt.rcParams['figure.figsize'] = (10, 7)

# 下载多资产数据
tickers = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'JPM', 'GLD', 'TLT']
prices = yf.download(tickers, start='2020-01-01', end='2024-01-01',
                     progress=False)['Close'].dropna()
returns = prices.pct_change().dropna()

n_assets = len(tickers)
mu = returns.mean() * 252           # 年化期望收益
cov = returns.cov() * 252            # 年化协方差矩阵

print(f'资产数量: {n_assets}')
print('年化期望收益率：')
print(mu.round(3))

## 1. 组合收益率与风险

$$\mu_p = \mathbf{w}^T \boldsymbol{\mu}, \quad \sigma_p = \sqrt{\mathbf{w}^T \boldsymbol{\Sigma} \mathbf{w}}$$

In [ ]:
def portfolio_stats(weights, mu, cov):
    """计算组合年化收益率、波动率、夏普比率"""
    w = np.array(weights)
    ret = w @ mu
    vol = np.sqrt(w @ cov @ w)
    sharpe = (ret - 0.04) / vol
    return ret, vol, sharpe

# 等权组合
equal_w = np.ones(n_assets) / n_assets
eq_ret, eq_vol, eq_sr = portfolio_stats(equal_w, mu, cov)
print(f'等权组合 → 收益: {eq_ret:.2%}, 波动: {eq_vol:.2%}, Sharpe: {eq_sr:.2f}')

## 2. 蒙特卡洛模拟 — 随机组合

In [ ]:
np.random.seed(42)
n_portfolios = 5000
results = np.zeros((n_portfolios, 3))
weights_list = []

for i in range(n_portfolios):
    w = np.random.random(n_assets)
    w /= w.sum()
    ret, vol, sr = portfolio_stats(w, mu, cov)
    results[i] = [ret, vol, sr]
    weights_list.append(w)

fig, ax = plt.subplots()
sc = ax.scatter(results[:, 1], results[:, 0], c=results[:, 2],
                cmap='viridis', alpha=0.5, s=10)
plt.colorbar(sc, label='Sharpe Ratio')
ax.scatter(eq_vol, eq_ret, marker='*', s=300, color='red', zorder=5, label='等权组合')
ax.set_xlabel('年化波动率')
ax.set_ylabel('年化收益率')
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
ax.set_title(f'{n_portfolios} 个随机组合 — 有效前沿雏形', fontsize=13)
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## 3. 优化求解最优组合

In [ ]:
constraints = ({'type': 'eq', 'fun': lambda w: np.sum(w) - 1})  # 权重之和=1
bounds = [(0, 1)] * n_assets  # 不允许做空
w0 = equal_w  # 初始猜测

# 最大夏普比率组合
def neg_sharpe(w):
    return -portfolio_stats(w, mu, cov)[2]

res_sr = minimize(neg_sharpe, w0, bounds=bounds, constraints=constraints)
w_sr = res_sr.x
sr_ret, sr_vol, sr_sharpe = portfolio_stats(w_sr, mu, cov)

# 最小方差组合
def portfolio_vol(w):
    return portfolio_stats(w, mu, cov)[1]

res_mv = minimize(portfolio_vol, w0, bounds=bounds, constraints=constraints)
w_mv = res_mv.x
mv_ret, mv_vol, mv_sharpe = portfolio_stats(w_mv, mu, cov)

# 可视化结果
fig, ax = plt.subplots()
sc = ax.scatter(results[:, 1], results[:, 0], c=results[:, 2],
                cmap='viridis', alpha=0.3, s=8, zorder=1)
plt.colorbar(sc, label='Sharpe Ratio')

ax.scatter(sr_vol, sr_ret, marker='*', s=400, color='gold', zorder=5,
            label=f'最大夏普 (SR={sr_sharpe:.2f})')
ax.scatter(mv_vol, mv_ret, marker='D', s=200, color='cyan', zorder=5,
            label=f'最小方差 (SR={mv_sharpe:.2f})')
ax.scatter(eq_vol, eq_ret, marker='o', s=200, color='red', zorder=5,
            label=f'等权组合 (SR={eq_sr:.2f})')

ax.set_xlabel('年化波动率')
ax.set_ylabel('年化收益率')
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
ax.set_title('有效前沿与最优组合', fontsize=13)
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

# 最大夏普组合的权重分布
print('\n最大夏普比率组合权重：')
for t, w in sorted(zip(tickers, w_sr), key=lambda x: -x[1]):
    print(f'  {t:<8}: {w:.2%}')

## 🎯 练习

1. 增加约束：单资产最大权重不超过 30%（`bounds = [(0, 0.3)] * n_assets`），观察权重变化。
2. 加入一只加密货币 ETF（如 `'BITO'`），重新计算有效前沿，高波动资产如何影响组合？
3. 用 PyPortfolioOpt 库实现同样的功能，比较结果差异：`pip install PyPortfolioOpt`

---
**下一节** → `02_risk_parity.ipynb`